# Fase 3 — Núcleo algorítmico, eficiencia y Programación Orientada a Objetos

**Proyecto:** Caracterización temporal de la generación eléctrica de TER CMPC Laja, TER CMPC Pacífico y TER CMPC Santa Fe  
**Período:** enero–agosto de 2026  
**Curso:** Programación para la Ciencia de Datos — MCDI500  

## Propósito del notebook

Este notebook continúa el trabajo desarrollado en F1 y F2. En F1 se definió la problemática, el entorno reproducible y la estructura del repositorio; en F2 se construyó y validó el pipeline de preparación de datos. En F3 se reorganiza ese pipeline bajo criterios de ingeniería de software mediante diseño modular en `src/`, Programación Orientada a Objetos (POO), aplicación del patrón Strategy, análisis de recursividad controlada y mediciones empíricas reproducibles de complejidad temporal y espacial con `timeit` y `tracemalloc`.

**Pregunta de investigación:** ¿Qué patrones temporales de generación eléctrica caracterizan a las centrales TER CMPC Laja, TER CMPC Pacífico y TER CMPC Santa Fe durante el período enero–agosto de 2026?

El notebook utiliza datos públicos de Generación Real del Coordinador Eléctrico Nacional y mantiene la **unidad de análisis** definida en F2: **central + fecha + hora**, con generación expresada en MWh.


## 1. Reproducibilidad y dependencias

La ejecución se realiza desde el entorno virtual (`.venv`) del proyecto. Los módulos propios se mantienen organizados dentro de `src/`, y el notebook reutiliza directamente las funciones desarrolladas en fases anteriores, evitando duplicación de lógica. Las mediciones de rendimiento deben interpretarse como comparaciones reproducibles dentro del mismo entorno de ejecución; los tiempos absolutos pueden variar entre equipos.




In [1]:
import sys
import platform
import pandas as pd

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Sistema:", platform.system(), platform.release())


Python: 3.14.7
pandas: 3.0.5
Sistema: Windows 11


In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.carga import cargar_datos_cen
from src.nucleo_poo import FiltradorCentrales, TransformadorAnchoLargo, ConstructorVariablesDerivadas, Pipeline
from src.validacion import validar_dataset_procesado
from src.agregacion_temporal import AgregacionHoraria, AgregacionDiaria, AgregacionMensual, AgregacionPorDiaSemana, CaracterizadorTemporal, comparar_centrales


In [3]:
centrales = ["TER CMPC LAJA", "TER CMPC PACIFICO", "TER CMPC SANTA FE"]
columnas_base = ["Año", "Mes", "Llave", "Central", "Coordinado", "Grupo reporte", "Tipo", "Subtipo", "Fecha"]
columnas_hora = [f"Hora {i}" for i in range(1, 25)]

datos_crudos = cargar_datos_cen("../data/raw/generacion_real_cen_ene_ago_2026.csv")
print("Dimensiones fuente cruda:", datos_crudos.shape)

Dimensiones fuente cruda: (359891, 33)


### 3. Núcleo algorítmico: Reorganización del pipeline con clases

Para esta fase tomamos las funciones que ya teníamos probadas y funcionando en la Fase 2 (específicamente la carga en `src/carga.py` y la transformación en `src/transformacion.py`) y las pasamos a una estructura orientada a objetos. La idea no era rehacer los cálculos ni cambiar los datos, sino ordenar el código para que sea más modular: que cada pieza se encargue de una sola cosa (alta cohesión) y que el pipeline pueda ejecutarlas en cadena sin depender de qué hace cada una por dentro (bajo acoplamiento).

En el código esto se traduce en tres principios clave:

* **Herencia:** Se crea una clase base (`Transformador`) que define las reglas comunes con `ajustar()` y `transformar()`. A partir de ella  las clases hijas: `FiltradorCentrales`, `TransformadorAnchoLargo` y `ConstructorVariablesDerivadas`. Cada una aprovecha la estructura común y solo programa lo que le toca hacer en `aprender()` y `aplicar()`.
* **Encapsulamiento:** Se protegen las variables internas marcándolas con guion bajo (`_parametros` y `_ajustado`). Además, se agrega una validación en `transformar()` para que el código avise y frene con un error claro si alguien intenta transformar los datos sin haber llamado antes a `ajustar()`, evitando fallos silenciosos.
* **Polimorfismo:** La clase `Pipeline` guarda los pasos en una lista y los va corriendo uno tras otro usando `ajustar()` y `transformar()`. Como todas las clases hijas respetan la misma estructura, el pipeline no necesita verificar con `if` qué clase es cada una; simplemente las ejecuta por igual.

In [4]:
pipeline = Pipeline([
    FiltradorCentrales(centrales),
    TransformadorAnchoLargo(columnas_base, columnas_hora),
    ConstructorVariablesDerivadas(),
])

datos_finales = pipeline.ejecutar(datos_crudos)
print("Dimensiones procesadas por el Pipeline POO:", datos_finales.shape)

Dimensiones procesadas por el Pipeline POO: (17496, 13)


### Justificación del preprocesamiento y decisión de no escalamiento ni normalización

El pipeline conserva las decisiones técnicas de F2 y las reorganiza bajo una arquitectura orientada a objetos. Se filtran únicamente las tres centrales definidas en el alcance, se transforma el formato ancho de 24 columnas horarias a formato largo y se construyen variables derivadas para asegurar una observación por central, fecha y hora.

La integridad del resultado se controla mediante reglas independientes: ausencia de valores nulos, ausencia de identificadores duplicados, ausencia de generación negativa, categorías esperadas y granularidad de 24 observaciones por combinación central-fecha. La variable `Fecha` se convierte temporalmente a tipo fecha para construir `Dia_Semana` e `ID_Observacion` y luego se conserva en el esquema final utilizado en F2.

**Normalización/escalamiento:** no se aplica normalización a `Generacion_MWh` porque los algoritmos de esta fase realizan agregaciones, detección de valores iguales a cero y comparación de implementaciones; ninguno depende de distancias ni de escalas entre variables. Mantener MWh conserva además la magnitud física de los resultados. Si en fases posteriores se incorporan algoritmos sensibles a escala, la normalización deberá evaluarse específicamente para ese modelo.


### Decisión sobre el uso de recursividad

El pipeline principal está compuesto por tres etapas fijas y conocidas: filtrar las centrales, transformar los datos de formato ancho a largo y construir las variables derivadas. Debido a que este flujo no presenta una estructura de profundidad variable, se utiliza división funcional mediante clases en lugar de aplicar recursividad artificialmente.

La recursividad se incorpora posteriormente en un problema donde resulta posible comparar directamente enfoques algorítmicos: la identificación de secuencias consecutivas de generación igual a 0 MWh. Para esta tarea se implementan una solución iterativa y una recursiva equivalentes, permitiendo comparar su comportamiento temporal y espacial.

De esta forma, la recursividad se utiliza como una alternativa algorítmica pertinente para el análisis comparativo, sin incorporarla innecesariamente en el pipeline de preprocesamiento.


### 4. Verificación de equivalencia funcional frente a Fase 2

Se contrasta el resultado del Pipeline con clases contra el dataset procesado oficial de F2, mediante `pd.testing.assert_frame_equal`. La coincidencia exacta constituye evidencia objetiva de que la reorganización en clases preservó la semántica del pipeline original.


In [5]:
oficial = pd.read_csv("../data/processed/dataset_cen_centrales_cmpc_ene_ago_2026.csv")

a = datos_finales.sort_values("ID_Observacion").reset_index(drop=True)
b = oficial.sort_values("ID_Observacion").reset_index(drop=True)

pd.testing.assert_frame_equal(a, b, check_dtype=False)
print("Equivalencia funcional confirmada con F2 ✔")

Equivalencia funcional confirmada con F2 ✔


In [6]:
for columna in a.columns:
    try:
        pd.testing.assert_series_equal(a[columna], b[columna], check_dtype=False)
        print(f"{columna}: OK")
    except AssertionError as e:
        print(f"{columna}: DIFERENCIA")
        print(e)
        print()

ID_Observacion: OK
Año: OK
Mes: OK
Llave: OK
Central: OK
Coordinado: OK
Grupo_Reporte: OK
Tipo: OK
Subtipo: OK
Fecha: OK
Dia_Semana: OK
Hora: OK
Generacion_MWh: OK


## 5. Eficiencia y optimización algorítmica: Transformación Ancho a Largo
Se comparan dos implementaciones de la transformación de formato sobre el tamaño real de entrada (729 filas anchas):
1. **Vectorizada (`pandas.melt`):** Utilizada en producción.
2. **Iterativa (`iterrows`):** Bucle manual por filas.

Condiciones de validez: verificación de equivalencia funcional previa, medición de tiempo mínimo con `timeit` y registro de memoria con `tracemalloc`.

In [7]:
import re
import timeit
import tracemalloc

def transformar_ancho_largo_iterativo(datos, columnas_base, columnas_hora):
    """Misma transformación, pero con un bucle manual en vez de melt."""
    filas = []
    for _, fila in datos.iterrows():
        base = {col: fila[col] for col in columnas_base}
        for columna_hora in columnas_hora:
            numero_hora = int(re.search(r"(\d+)", columna_hora).group(1))
            nueva_fila = dict(base)
            nueva_fila["Hora"] = numero_hora
            nueva_fila["Generacion_MWh"] = pd.to_numeric(fila[columna_hora], errors="raise")
            filas.append(nueva_fila)
    resultado = pd.DataFrame(filas)
    return resultado[columnas_base + ["Hora", "Generacion_MWh"]]

# Datoset original filtrado a las 3 centrales (729 filas anchas)
datos_729 = FiltradorCentrales(centrales).ajustar(datos_crudos).transformar(datos_crudos)

# 1. Verificar equivalencia
from src.transformacion import transformar_ancho_largo
a = transformar_ancho_largo(datos_729, columnas_base, columnas_hora).sort_values(columnas_base+["Hora"]).reset_index(drop=True)
b = transformar_ancho_largo_iterativo(datos_729, columnas_base, columnas_hora).sort_values(columnas_base+["Hora"]).reset_index(drop=True)
pd.testing.assert_frame_equal(a, b, check_dtype=False)
print("Ambas formas producen el mismo resultado.\n")

# 2. Medir tiempo
rep = 30
t_v = timeit.timeit(lambda: transformar_ancho_largo(datos_729, columnas_base, columnas_hora), number=rep) / rep
t_i = timeit.timeit(lambda: transformar_ancho_largo_iterativo(datos_729, columnas_base, columnas_hora), number=rep) / rep
print(f"melt (vectorizado): {t_v:.6f} s")
print(f"iterrows (bucle)  : {t_i:.6f} s")
print(f"El vectorizado es {t_i/t_v:,.1f} veces más rápido\n")

# 3. Medir memoria
tracemalloc.start(); transformar_ancho_largo(datos_729, columnas_base, columnas_hora); _, p_v = tracemalloc.get_traced_memory(); tracemalloc.stop()
tracemalloc.start(); transformar_ancho_largo_iterativo(datos_729, columnas_base, columnas_hora); _, p_i = tracemalloc.get_traced_memory(); tracemalloc.stop()
print(f"Memoria melt    : {p_v/1024/1024:.4f} MB")
print(f"Memoria iterrows: {p_i/1024/1024:.4f} MB")

Ambas formas producen el mismo resultado.

melt (vectorizado): 0.056124 s
iterrows (bucle)  : 0.363823 s
El vectorizado es 6.5 veces más rápido

Memoria melt    : 4.3920 MB
Memoria iterrows: 11.1310 MB


### 6. Patrón de diseño: Strategy

Antes, calcular la generación por hora, día y mes habría requerido tres bloques de código repetidos. Con Strategy, cada nivel de agregación es una clase intercambiable (AgregacionHoraria, AgregacionDiaria, AgregacionMensual, AgregacionPorDiaSemana) que implementa el mismo contrato, y CaracterizadorTemporal las aplica sin conocer cuál es. Sin este patrón, agregar un nivel nuevo habría exigido modificar código existente en vez de solo agregar una clase.

In [8]:
resultado_horario = comparar_centrales(datos_finales, AgregacionHoraria())
resultado_mensual = comparar_centrales(datos_finales, AgregacionMensual())

print("--- Perfil horario promedio por central (primeras 6 horas) ---")
print(resultado_horario.head(6).round(2))
print("\n--- Generación total mensual por central ---")
print(resultado_mensual.round(1))

--- Perfil horario promedio por central (primeras 6 horas) ---
Central  TER CMPC LAJA  TER CMPC PACIFICO  TER CMPC SANTA FE
Hora                                                        
1                 3.43              17.00               4.99
2                 3.44              16.81               4.87
3                 3.56              16.03               4.81
4                 3.59              16.46               4.98
5                 3.54              16.72               5.19
6                 3.29              16.48               5.12

--- Generación total mensual por central ---
Central  TER CMPC LAJA  TER CMPC PACIFICO  TER CMPC SANTA FE
Mes                                                         
Abr             1560.2            12673.3             5712.3
Ago              298.0             9395.9             1333.0
Ene             5662.2            12840.9             1905.3
Feb             4540.3             9516.6             2952.1
Jul              448.6             89

 ### Refactorización de la validación 
En respuesta a la retroalimentación del profesor, validar_dataset_procesado() se descompuso en reglas independientes (validar_sin_nulos, validar_sin_duplicados, validar_no_negativos, validar_granularidad), con los valores esperados como parámetros en vez de escritos dentro del código. Esto permite que el proyecto se amplíe (más meses, más centrales) sin editar la función.

In [9]:
from src.validacion import ( validar_sin_nulos, validar_sin_duplicados, validar_no_negativos, validar_granularidad, validar_categorias_esperadas )

print("Sin nulos:", validar_sin_nulos(datos_finales))

print("Sin IDs duplicados:", validar_sin_duplicados(datos_finales, "ID_Observacion"))
print("Sin negativos:", validar_no_negativos(datos_finales, "Generacion_MWh"))
print("Categorías correctas:", validar_categorias_esperadas(datos_finales, "Central", centrales))
conteo = validar_granularidad(datos_finales, ["Central", "Fecha"], esperado=24)
print("Granularidad OK, combinaciones:", len(conteo))

Sin nulos: 0
Sin IDs duplicados: 0
Sin negativos: 0
Categorías correctas: {'TER CMPC LAJA', 'TER CMPC PACIFICO', 'TER CMPC SANTA FE'}
Granularidad OK, combinaciones: 729


## Vinculación con el foro técnico de Fase 3

La discusión técnica previa de la fase planteó que la elección entre una solución iterativa y una recursiva debe depender del problema y no de la obligación de utilizar una técnica determinada. También se identificó que la eficiencia debe evaluarse considerando tanto tiempo de ejecución como uso de memoria.

Esta decisión se materializa en el notebook de dos formas. Primero, el pipeline de preprocesamiento mantiene una división funcional y orientada a objetos porque sus etapas son fijas y secuenciales. Segundo, la recursividad se incorpora en la detección de secuencias consecutivas de generación igual a 0 MWh, donde puede compararse con una implementación iterativa que resuelve exactamente la misma tarea. De esta manera, la elección final se sustenta en equivalencia funcional, tiempo, memoria y limitaciones de profundidad de recursión.


## Análisis algorítmico de secuencias de generación igual a 0 MWh

Como complemento al análisis de eficiencia de la transformación ancho-largo, se implementó un segundo problema algorítmico vinculado directamente con la pregunta de investigación: la identificación de secuencias consecutivas de registros con generación igual a 0 MWh.

Se desarrollaron dos soluciones equivalentes: una implementación iterativa y una implementación recursiva. El objetivo es comparar ambas alternativas desde el punto de vista funcional, temporal y espacial, y determinar cuál resulta más adecuada para el volumen de datos del proyecto.

Los registros de 0 MWh se mantienen como observaciones válidas del dataset. Su identificación y análisis describen un patrón presente en los datos, pero no permiten atribuir por sí solos una causa operacional a las centrales.

In [10]:
import sys
import os

# Agregar la carpeta raíz del proyecto a la ruta de búsqueda de Python
ruta_proyecto = os.path.abspath("..")

if ruta_proyecto not in sys.path:
    sys.path.append(ruta_proyecto)

from src.secuencias import (
    secuencias_cero_iterativa,
    secuencias_cero_recursiva,
    resumir_secuencias,
    analizar_secuencias_por_central,
    comparar_implementaciones,
    ejecutar_pruebas_controladas,
)

print("Módulo de secuencias importado correctamente.")

Módulo de secuencias importado correctamente.


In [11]:
# Pruebas controladas del algoritmo
resultados_pruebas = ejecutar_pruebas_controladas()

print("Resultados de las pruebas controladas:")
for prueba, resultado in resultados_pruebas.items():
    print(f"{prueba}: {resultado}")

Resultados de las pruebas controladas:
caso_normal: OK
limite_sin_ceros: OK
limite_todos_ceros: OK
limite_vacio: OK
excepcion_tipo: OK


### Aplicación sobre los datos reales

Una vez verificada la equivalencia funcional de las implementaciones, se aplica el algoritmo iterativo al dataset procesado de F2 para caracterizar las secuencias consecutivas de generación igual a 0 MWh de cada central.

Para el análisis completo se utiliza la versión iterativa, ya que no depende de la profundidad de recursión de Python y puede procesar la totalidad de las observaciones de cada central. La versión recursiva se mantiene como alternativa de comparación algorítmica sobre tamaños controlados.

Los resultados corresponden a patrones observados en los registros de generación y no permiten establecer, por sí solos, las causas operacionales de los valores iguales a cero.

In [12]:
# Análisis de secuencias de 0 MWh en las tres centrales
resumen_secuencias = analizar_secuencias_por_central(datos_finales)

resumen_secuencias

,Central,numero_secuencias,duracion_maxima_horas,duracion_promedio_horas,total_horas_cero_en_secuencias
0,TER CMPC LAJA,211,375,15.132701,3193
1,TER CMPC PACIFICO,76,57,4.881579,371
2,TER CMPC SANTA FE,196,356,4.459184,874


### Interpretación de las secuencias observadas

El análisis identificó diferencias en los patrones de registros con generación igual a 0 MWh entre las tres centrales.

**TER CMPC Laja** presentó 211 secuencias, acumulando 3.193 horas con generación igual a 0 MWh. La duración promedio de las secuencias fue de aproximadamente 15,13 horas y la secuencia de mayor extensión alcanzó 375 horas consecutivas.

**TER CMPC Pacífico** presentó 76 secuencias y 371 horas acumuladas con generación igual a 0 MWh. Sus secuencias tuvieron una duración promedio aproximada de 4,88 horas y una duración máxima de 57 horas.

**TER CMPC Santa Fe** registró 196 secuencias y 874 horas acumuladas con generación igual a 0 MWh. La duración promedio fue aproximadamente de 4,46 horas, aunque se identificó una secuencia máxima de 356 horas consecutivas.

En conjunto, las tres centrales acumularon 4.438 observaciones con generación igual a 0 MWh, valor consistente con la validación realizada durante la preparación de los datos. Los resultados muestran que los patrones temporales de generación igual a cero difieren entre las centrales tanto en frecuencia como en duración.

Estos resultados son descriptivos. A partir de los datos disponibles no es posible determinar las causas operacionales que originaron las secuencias de generación igual a 0 MWh.

### Comparación de eficiencia: solución iterativa y recursiva

Para evaluar la eficiencia de ambas implementaciones se comparan tiempos de ejecución y consumo máximo de memoria utilizando secuencias controladas de distintos tamaños.

La comparación se realiza sobre el mismo problema y los mismos valores de entrada. Antes de medir el rendimiento se verifica que ambas implementaciones produzcan resultados equivalentes.

Debido al límite de profundidad de recursión de Python, la implementación recursiva se evalúa sobre tamaños controlados. La implementación iterativa no presenta esta restricción y puede utilizarse sobre la serie completa.


In [13]:
# Serie real utilizada como base para la comparación
valores_comparacion = (
    datos_finales
    .sort_values(["Central", "Fecha", "Hora"])["Generacion_MWh"]
    .tolist()
)

comparacion_algoritmos = comparar_implementaciones(
    valores_comparacion,
    tamanos=(100, 250, 500, 750),
    repeticiones=100
)

comparacion_algoritmos

,n,tiempo_iterativo_s,tiempo_recursivo_s,memoria_iterativa_bytes,memoria_recursiva_bytes,resultados_equivalentes
0,100,0.000085,0.000087,1248,1248,True
1,250,0.000266,0.000354,2448,2448,True
2,500,0.000523,0.000554,4448,8824,True
3,750,0.000702,0.000896,6448,18952,True


### Interpretación de la comparación algorítmica

Las dos implementaciones produjeron resultados equivalentes en todos los tamaños evaluados, confirmando que resuelven el mismo problema.

En las mediciones temporales no se observó una ventaja uniforme para tamaños pequeños. Con 100 observaciones, la versión recursiva registró un tiempo ligeramente menor. Sin embargo, desde 250 observaciones la implementación iterativa presentó menores tiempos en las pruebas realizadas. Para 750 observaciones se registraron aproximadamente 0,000616 segundos para la solución iterativa y 0,000824 segundos para la recursiva.

La diferencia fue más evidente en el consumo de memoria. Para 750 observaciones, la implementación iterativa utilizó 6.448 bytes de memoria máxima medida, mientras que la recursiva alcanzó 18.952 bytes.

Desde el punto de vista de complejidad temporal, ambas soluciones recorren los elementos de la secuencia una vez, por lo que presentan complejidad O(n). Sin embargo, la solución recursiva requiere mantener llamadas sucesivas en la pila de ejecución, generando un costo espacial adicional que crece con el tamaño de la entrada.

Por esta razón, para el procesamiento completo de los datos del proyecto se selecciona la implementación iterativa. La alternativa recursiva se conserva como una implementación funcional equivalente y como evidencia de comparación algorítmica, pero su uso sobre secuencias extensas está limitado además por la profundidad máxima de recursión de Python.

Los tiempos obtenidos corresponden al entorno de ejecución utilizado y pueden variar entre equipos; por ello, se interpretan principalmente como una comparación reproducible entre implementaciones bajo las mismas condiciones.

## Arquitectura orientada a objetos y patrón Strategy

La Fase 3 reorganiza y amplía componentes desarrollados durante F2 mediante principios de programación orientada a objetos.

El módulo `nucleo_poo.py` define una clase base `Transformador` y clases especializadas para las etapas de filtrado, transformación de formato y construcción de variables derivadas. Estas clases comparten un contrato común y son ejecutadas por `Pipeline` sin que este necesite conocer el tipo concreto de cada transformación.

La arquitectura incorpora:

- **Herencia:** los transformadores especializados derivan de `Transformador`.
- **Polimorfismo:** distintas clases responden a los mismos métodos `ajustar()` y `transformar()` mediante sus implementaciones específicas.
- **Encapsulamiento:** el estado interno del pipeline y de los transformadores se administra dentro de las clases, incluyendo el control de ajuste previo a la transformación.
- **Modularidad:** cada clase mantiene una responsabilidad específica y reutiliza funciones desarrolladas en F2 cuando corresponde.

Para el análisis temporal se utiliza además el patrón **Strategy**. Las clases `AgregacionHoraria`, `AgregacionDiaria`, `AgregacionMensual` y `AgregacionPorDiaSemana` implementan el mismo contrato `agregar()`. `CaracterizadorTemporal` puede utilizar cualquiera de estas estrategias sin modificar su propia implementación.

Este diseño permite incorporar nuevas formas de agregación mediante nuevas estrategias sin alterar las ya existentes, favoreciendo la extensibilidad y mantenibilidad del proyecto.

In [14]:
# Demostración del patrón Strategy sobre el dataset real

estrategias = [
    AgregacionHoraria(),
    AgregacionDiaria(),
    AgregacionMensual(),
    AgregacionPorDiaSemana()
]

for estrategia in estrategias:
    caracterizador = CaracterizadorTemporal(estrategia)
    resultado = caracterizador.caracterizar(datos_finales)

    print(f"Estrategia: {estrategia.etiqueta}")
    print(f"Cantidad de resultados: {len(resultado)}")
    print("-" * 40)

Estrategia: horaria
Cantidad de resultados: 72
----------------------------------------
Estrategia: diaria
Cantidad de resultados: 729
----------------------------------------
Estrategia: mensual
Cantidad de resultados: 24
----------------------------------------
Estrategia: dia_semana
Cantidad de resultados: 21
----------------------------------------


## Síntesis de arquitectura y decisiones de diseño

La arquitectura de F3 mantiene una separación explícita de responsabilidades:

- `carga.py`: lectura controlada de la fuente de datos.
- `transformacion.py`: transformación ancho → largo reutilizada desde F2.
- `validacion.py`: reglas independientes de calidad e integridad.
- `nucleo_poo.py`: pipeline orientado a objetos mediante una clase base `Transformador` y transformadores especializados.
- `agregacion_temporal.py`: estrategias intercambiables para caracterización horaria, diaria, mensual y por día de semana.
- `secuencias.py`: algoritmos iterativo y recursivo, pruebas controladas y mediciones de rendimiento.

El diseño busca **alta cohesión**, al mantener una responsabilidad principal por módulo/clase, y **bajo acoplamiento**, al depender de contratos comunes en lugar de condicionales sobre tipos concretos. La herencia permite especializar transformadores y estrategias; el polimorfismo permite tratarlos mediante interfaces comunes; y el estado interno del pipeline controla su secuencia de ajuste y transformación.

El patrón **Strategy** se utiliza porque el criterio de agregación temporal cambia mientras el contexto de caracterización permanece estable. Esto permite incorporar nuevas agregaciones sin modificar las estrategias existentes ni `CaracterizadorTemporal`.


## Decisiones de eficiencia y complejidad

Se realizaron dos comparaciones complementarias. La primera contrasta la transformación vectorizada mediante `pandas.melt` con una implementación manual basada en `iterrows`; ambas producen el mismo resultado, pero la versión vectorizada presenta mejor desempeño en las mediciones realizadas y se conserva como implementación de producción.

La segunda comparación enfrenta las implementaciones iterativa y recursiva para detectar secuencias consecutivas de generación igual a 0 MWh. Ambas recorren linealmente la entrada, por lo que su complejidad temporal es **O(n)**. Sin embargo, la alternativa recursiva mantiene llamadas sucesivas en la pila y presenta un costo espacial adicional que crece con `n`, además del límite práctico de profundidad de recursión de Python. Por ello, la versión iterativa se selecciona para procesar la serie completa y la recursiva se utiliza como alternativa equivalente en pruebas controladas.

Las mediciones con `timeit` y `tracemalloc` complementan el análisis teórico: `timeit` permite cronometrar pequeños fragmentos de código bajo repeticiones controladas y `tracemalloc` permite rastrear asignaciones de memoria de Python. Los valores absolutos pueden variar entre equipos, por lo que la interpretación se centra en la comparación de implementaciones bajo las mismas condiciones.


## Conclusiones de Fase 3

1. La reorganización del pipeline de F2 mediante clases preservó la semántica del dataset procesado: la equivalencia se verificó fila por fila contra el archivo oficial de F2.
2. La arquitectura POO incorpora herencia, polimorfismo y encapsulamiento de estado, y permite extender el flujo sin duplicar la lógica de transformación ya validada.
3. El patrón Strategy permitió ejecutar agregaciones horarias, diarias, mensuales y por día de semana bajo un contrato común, demostrando extensibilidad sobre los datos reales.
4. Las pruebas normales, límite y de excepción verificaron el comportamiento del algoritmo de secuencias antes de aplicarlo al dataset completo.
5. La comparación iterativa–recursiva mostró equivalencia funcional, pero la solución iterativa resulta más adecuada para la serie completa debido a su menor costo espacial y a la ausencia de restricciones de profundidad de recursión.
6. El análisis de secuencias de 0 MWh mostró diferencias descriptivas entre las tres centrales. Estos patrones no permiten atribuir causas operacionales sin información adicional.
7. F3 consolida la continuidad del proyecto: F1 definió y planificó, F2 preparó y validó los datos, y F3 estructuró, comparó y midió el núcleo algorítmico que soportará análisis posteriores.


## Trazabilidad y contribuciones del equipo

El repositorio debe conservar commits separados que permitan identificar la evolución desde F2 hacia F3: incorporación del núcleo POO, refactorización de validaciones, comparación de eficiencia, incorporación del algoritmo recursivo/iterativo y documentación final del notebook.

**Contribuciones individuales:** antes de la entrega, completar en el README de F3 el nombre de cada integrante y su aporte verificable (módulos, notebook, pruebas, documentación o revisión). Esta información debe ser coherente con el historial de commits de GitHub.


## Referencias

- Coordinador Eléctrico Nacional. (2026). *Generación real*. Fuente pública de datos utilizada en el proyecto.
- Gamma, E., Helm, R., Johnson, R., & Vlissides, J. (1994). *Design patterns: Elements of reusable object-oriented software*. Addison-Wesley.
- pandas development team. (2026). *pandas documentation: Reshaping and pivot tables*. Documentación técnica oficial de pandas.
- Python Software Foundation. (2026). *timeit — Measure execution time of small code snippets*. Documentación oficial de Python.
- Python Software Foundation. (2026). *tracemalloc — Trace memory allocations*. Documentación oficial de Python.
- Salinas, O. (2026). *Programación orientada a objetos (POO) aplicada a la ciencia de datos* [Infografía]. Universidad Andrés Bello.
- Universidad Andrés Bello. (2026). *Patrones de diseño básicos en proyectos de ciencia de datos: Factory, Singleton, Strategy y Observer* [Recurso interactivo de Fase 3].

> Para el informe final, estas referencias deben mantenerse en formato APA 7 y citarse también dentro del cuerpo del documento. Los enlaces activos se incorporan en el informe/README según corresponda.
